# 02. Baseline: Bag-of-Words Logistic Regression on SST-2

The simplest, fully transparent baseline: a **TF-IDF bag-of-words** representation and a
**logistic regression** classifier. No GPU is needed, but the notebook keeps the same
Colab setup as the others so everything runs in one place.

Why this baseline matters for the BAMIC paper: it is the classical *interpretable*
statistical model. Its coefficients give one global weight per word, which we can later
contrast with BAMIC's *per-document, per-word posterior* sentiment contributions.

Steps: setup → load shared splits → TF-IDF features → fit logistic regression →
metrics (accuracy, F1, AUC, Brier, NLL, ECE) → inspect top words → save results.

In [1]:
# ================================================================
# 0. Check the Colab GPU
# ================================================================

# Ask the Colab runtime which GPU we were given.
gpu_info = !nvidia-smi
# Join the command output lines into one printable string.
gpu_info = '\n'.join(gpu_info)
# If the command failed, we are not connected to a GPU runtime.
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU. In Colab: Runtime -> Change runtime type -> T4 GPU.')
else:
    print(gpu_info)

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# ================================================================
# 1. Check the Colab RAM
# ================================================================

# psutil reports how much system memory this runtime has.
import psutil
# Convert bytes to gigabytes for a readable number.
ram_gb = psutil.virtual_memory().total / 1e9
# Print the available RAM so we know which runtime type we received.
print('Your runtime has {:.1f} gigabytes of available RAM'.format(ram_gb))

Your runtime has 13.6 gigabytes of available RAM


In [3]:
# ================================================================
# 2. Basic imports and reproducibility
# ================================================================

# pathlib gives clean, operating-system-safe file paths.
from pathlib import Path

# random controls Python-level randomness.
import random

# numpy handles numeric arrays.
import numpy as np

# pandas handles CSV data tables.
import pandas as pd

# TfidfVectorizer turns raw text into sparse bag-of-words features.
from sklearn.feature_extraction.text import TfidfVectorizer

# LogisticRegression is the linear classifier for this baseline.
from sklearn.linear_model import LogisticRegression

# Metrics for classification quality and calibration.
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, brier_score_loss, log_loss

# This seed keeps runs reproducible (same seed as the wine notebooks).
SEED = 20260526

# Seed Python's random module.
random.seed(SEED)

# Seed NumPy's random generator.
np.random.seed(SEED)

In [4]:
# ================================================================
# 3. Mount Google Drive and define project paths
# ================================================================

# Mount Google Drive so Colab can read and write project files.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

# This is the same Google Drive project folder used by the wine BAMIC notebooks.
# If your folder name is different in Drive, edit only this line.
PROJECT_DIR = Path('/content/drive/MyDrive/AMIC project')

# All SST-2 benchmark files live inside this subfolder.
BENCH_DIR = PROJECT_DIR / 'sst2_benchmark'

# The shared, prepared SST-2 splits are stored here by notebook 00.
DATA_DIR = BENCH_DIR / 'data'

# This notebook writes all of its results into its own output folder.
EXPERIMENT_NAME = 'sst2_logreg_bow'
OUTPUT_DIR = BENCH_DIR / 'outputs' / EXPERIMENT_NAME

# Create the output folder if it does not exist yet.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Print the paths so we can verify them before loading data.
print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR   :', DATA_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)

Mounted at /content/drive
Google Drive mounted.
PROJECT_DIR: /content/drive/MyDrive/AMIC project
DATA_DIR   : /content/drive/MyDrive/AMIC project/sst2_benchmark/data
OUTPUT_DIR : /content/drive/MyDrive/AMIC project/sst2_benchmark/outputs/sst2_logreg_bow


In [5]:
# ================================================================
# 4. Load the shared SST-2 splits prepared by notebook 00
# ================================================================

# Stop early with a clear message if notebook 00 has not been run yet.
for name in ['sst2_train.csv', 'sst2_valid.csv', 'sst2_test.csv']:
    if not (DATA_DIR / name).exists():
        raise FileNotFoundError(f'Missing {DATA_DIR / name}. Run 00_prepare_sst2_data.ipynb first.')

# Read the three prepared splits from Drive.
train_df = pd.read_csv(DATA_DIR / 'sst2_train.csv')
valid_df = pd.read_csv(DATA_DIR / 'sst2_valid.csv')
test_df = pd.read_csv(DATA_DIR / 'sst2_test.csv')

# Force the text column to string in case pandas parsed something unusually.
for df in [train_df, valid_df, test_df]:
    df['text'] = df['text'].astype(str)

# Print shapes and label balance to confirm the data looks right.
print('train:', train_df.shape, '| positive rate:', round(train_df['y'].mean(), 4))
print('valid:', valid_df.shape, '| positive rate:', round(valid_df['y'].mean(), 4))
print('test :', test_df.shape, '| positive rate:', round(test_df['y'].mean(), 4))

# Show two example rows so we can see what the model will read.
train_df.head(2)

train: (63981, 2) | positive rate: 0.5578
valid: (3368, 2) | positive rate: 0.5579
test : (872, 2) | positive rate: 0.5092


,text,y
0,was n't one of the film 's virtues,0
1,elegant work,1


In [6]:
# ================================================================
# 5. TF-IDF bag-of-words features
# ================================================================

# The vectorizer lowercases text, builds the vocabulary from TRAINING data only,
# and weights each word count by its inverse document frequency.
vectorizer = TfidfVectorizer(
    lowercase=True,        # treat 'Good' and 'good' as the same feature
    ngram_range=(1, 2),    # use single words and two-word phrases (captures 'not good')
    min_df=2,              # a term must appear in at least 2 training sentences
    max_features=50000,    # cap the vocabulary so the model stays small and fast
)

# Fit the vocabulary on training text and transform it into a sparse matrix.
X_train = vectorizer.fit_transform(train_df['text'])

# Transform validation and test with the SAME fitted vocabulary (no leakage).
X_valid = vectorizer.transform(valid_df['text'])
X_test = vectorizer.transform(test_df['text'])

# Pull out the label arrays.
y_train_true = train_df['y'].to_numpy()
y_valid_true = valid_df['y'].to_numpy()
y_test_true = test_df['y'].to_numpy()

# Print the feature matrix shapes: (sentences, vocabulary terms).
print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test :', X_test.shape)

X_train: (63981, 50000)
X_valid: (3368, 50000)
X_test : (872, 50000)


In [7]:
# ================================================================
# 6. Fit the logistic regression classifier
# ================================================================

# C is the inverse regularization strength; 1.0 is the standard default.
# liblinear is a reliable solver for sparse TF-IDF problems of this size.
clf = LogisticRegression(C=1.0, solver='liblinear', random_state=SEED, max_iter=1000)

# Fit the classifier on the training features and labels.
clf.fit(X_train, y_train_true)

# predict_proba returns [P(negative), P(positive)]; keep the positive column.
train_probs = clf.predict_proba(X_train)[:, 1]
valid_probs = clf.predict_proba(X_valid)[:, 1]
test_probs = clf.predict_proba(X_test)[:, 1]

# Quick sanity check: training accuracy at the 0.5 threshold.
print('train accuracy:', round(accuracy_score(y_train_true, (train_probs >= 0.5).astype(int)), 4))

train accuracy: 0.9449


In [8]:
# ================================================================
# Metric helpers (identical to the wine BAMIC notebook, for fair comparison)
# ================================================================

def binary_metrics_from_probs(probs, labels, threshold=0.5):
    """Compute common binary metrics from predicted positive-class probabilities."""
    # Turn probabilities into hard 0/1 predictions at the given threshold.
    pred = (probs >= threshold).astype(int)
    # Fraction of documents classified correctly.
    acc = accuracy_score(labels, pred)
    # Harmonic mean of precision and recall for the positive class.
    f1 = f1_score(labels, pred, zero_division=0)
    # Threshold-free ranking quality; needs both classes present.
    auc = roc_auc_score(labels, probs) if len(np.unique(labels)) == 2 else np.nan
    # Mean squared error between probabilities and true labels.
    brier = brier_score_loss(labels, probs)
    # Negative log-likelihood of the true labels under the predicted probabilities.
    nll = log_loss(labels, np.clip(probs, 1e-7, 1 - 1e-7), labels=[0, 1])
    # Return everything in one dictionary.
    return {'acc': acc, 'f1': f1, 'auc': auc, 'brier': brier, 'nll': nll}


def expected_calibration_error(probs, labels, n_bins=10):
    """Compute a simple expected calibration error (ECE)."""
    # Create equally spaced probability bins between 0 and 1.
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    # Accumulate the weighted calibration gap here.
    ece = 0.0
    # Loop over each bin edge pair.
    for lo, hi in zip(bins[:-1], bins[1:]):
        # Include the right edge for the final bin.
        if hi == 1.0:
            mask = (probs >= lo) & (probs <= hi)
        else:
            mask = (probs >= lo) & (probs < hi)
        # Skip bins that contain no predictions.
        if not np.any(mask):
            continue
        # Average predicted probability inside this bin.
        conf = probs[mask].mean()
        # Empirical positive rate inside this bin.
        acc = labels[mask].mean()
        # Weight the |confidence - accuracy| gap by the bin frequency.
        ece += np.abs(conf - acc) * mask.mean()
    # Return the scalar ECE value.
    return float(ece)

In [9]:
# ================================================================
# Save final metrics and predictions to Drive
# ================================================================

# Collect one metrics row per split so the comparison notebook can read them later.
final_metric_rows = []

# Loop over the three splits with their predicted probabilities and true labels.
for split_name, probs, labels in [('train', train_probs, y_train_true),
                                  ('valid', valid_probs, y_valid_true),
                                  ('test', test_probs, y_test_true)]:
    # Compute accuracy, F1, AUC, Brier, and NLL for this split.
    metrics = binary_metrics_from_probs(probs, labels)
    # Compute the calibration error for this split.
    ece = expected_calibration_error(probs, labels)
    # Store everything in one row.
    final_metric_rows.append({'split': split_name, **metrics, 'ece': ece})
    # Print the row so we can see the result immediately.
    print(split_name, {k: round(v, 4) for k, v in metrics.items()}, 'ece=', round(ece, 4))

# Convert the rows into a small dataframe.
final_metrics_df = pd.DataFrame(final_metric_rows)

# Save the metrics table into this notebook's output folder.
final_metrics_df.to_csv(OUTPUT_DIR / 'final_metrics.csv', index=False)

# Also save the raw test predictions for later error analysis.
pd.DataFrame({'y_true': y_test_true, 'p_positive': test_probs}).to_csv(
    OUTPUT_DIR / 'test_predictions.csv', index=False)

# Confirm where everything was written.
print('Saved final_metrics.csv and test_predictions.csv to:', OUTPUT_DIR)

train {'acc': 0.9449, 'f1': 0.951, 'auc': np.float64(0.9862), 'brier': np.float64(0.0693), 'nll': 0.2705} ece= 0.1522
valid {'acc': 0.9103, 'f1': 0.9208, 'auc': np.float64(0.967), 'brier': np.float64(0.0865), 'nll': 0.3097} ece= 0.1264
test {'acc': 0.8131, 'f1': 0.8242, 'auc': np.float64(0.9081), 'brier': np.float64(0.1279), 'nll': 0.4027} ece= 0.0749
Saved final_metrics.csv and test_predictions.csv to: /content/drive/MyDrive/AMIC project/sst2_benchmark/outputs/sst2_logreg_bow


In [10]:
# ================================================================
# 8. Inspect the most positive and most negative words
# ================================================================

# Get the term corresponding to every coefficient position.
feature_names = np.array(vectorizer.get_feature_names_out())

# The linear coefficients: positive values push toward the positive class.
coefs = clf.coef_.ravel()

# Sort coefficient positions from most negative to most positive.
order = np.argsort(coefs)

# Build a small table of the 15 strongest negative and positive terms.
top_words = pd.DataFrame({
    'most_negative_term': feature_names[order[:15]],
    'negative_coef': np.round(coefs[order[:15]], 3),
    'most_positive_term': feature_names[order[-15:]][::-1],
    'positive_coef': np.round(coefs[order[-15:]][::-1], 3),
})

# Save the table so it can be cited as the classical-interpretability reference point.
top_words.to_csv(OUTPUT_DIR / 'top_terms.csv', index=False)

# Display the table in the notebook.
top_words

,most_negative_term,negative_coef,most_positive_term,positive_coef
0,too,-6.810,good,5.993
1,bad,-6.687,best,4.982
2,no,-5.654,fun,4.864
3,not,-5.424,entertaining,4.799
4,worst,-5.076,beautiful,4.642
5,dull,-4.886,powerful,4.598
6,lacks,-4.812,fascinating,4.561
7,less,-4.800,love,4.514
8,nothing,-4.328,enjoyable,4.503
9,mess,-4.270,funny,4.405


**Interpretation note for the paper.** Logistic regression gives one *global* weight per
term. BAMIC instead gives each word a *per-document* selection probability and sentiment
score **with posterior uncertainty**. Comparing the two illustrates exactly what the
Bayesian, context-dependent decomposition adds.